**Advanced Machine Learning (Semester 2 2025)**
# 8 Reinforcement Learning


*N. Hernitschek, 2025*


This Jupyter notebook gives an intro to Reinforcement Learning.



---
## Contents

* [Reinforcement Learning](#first-bullet)
* [Reinforcement Learning with `gymnasium` Environment](#second-bullet)
* [Custom `gymnasium` Environments](#third-bullet)
* [Summary](#fourth-bullet)


## 1. Reinforcement Learning<a class="anchor" id="first-bullet"></a>

Machine learning methods we have seen so far either fall into the category of supervised or unsupervised algorithms.
Reinforcement Learning stands out because it is used to train models in a live environment.

In reinforcement learning, the classic “agent-environment loop” pictured below represents how learning happens in Reinforcement Learning. 


<img src="../images/reinforcement_learning_cycle.png" alt="reinforcement" class="bg-primary" width="600px">


It’s simpler than it might first appear:

1.    Agent observes the current situation (like looking at a game screen)

2.    Agent chooses an action based on what it sees (like pressing a button)

3.    Environment responds with a new situation and a reward (game state changes, score updates)

4.    Repeat until the episode ends









## 2. Reinforcement Learning with `gymnasium` Environment <a class="anchor" id="second-bullet"></a>

OpenAI `gymnasium` is an open-source Python library for developing and comparing reinforcement learning algorithms by providing a standard API to communicate between learning algorithms and environments, as well as a standard set of environments compliant with that API. Since its release, this API (and the earlier version `gym`) has become the field standard.

You can find more information at 

https://gymnasium.farama.org/


OpenAI `gymnasium` comes with standard test environments for Reinforcement Learning, such as simple computer games like "Space Invaders", or simulations like balancing a pole on a moving cart. Whereas such environments can be fun to try out and can give an idea on the performance of various Reinforcement Learning algorithms, this can be limiting.
In addition to this, it is possible to build custom Reinforcement Learning environments using OpenAI `gymnasium`.

First we install the library:





In [ ]:
pip install gymnasium

### 2.1 The CartPole environment

The CartPole environment simulates balance a pole on a moving cart.
This example is good for illustrative purposes, as of:
* Simple but not trivial
* Fast training
* Clear success/failure criteria

We now continue with a first example. This is showing just the environment. **No training takes place here.**

When you run the code in the next cell, you should see a window showing a cart with a pole. The cart moves randomly left and right, and the pole eventually falls over. This is expected - without training, the agent is acting randomly.


In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt

matplotlib.rcParams.update({'font.size': 22})

import random

# Create our training environment - a cart with a pole that needs balancing
env = gym.make("CartPole-v1", render_mode="human")

# Reset environment to start a new episode
observation, info = env.reset()
# observation: what the agent can "see" - cart position, velocity, pole angle, etc.
# info: extra debugging information (usually not needed for basic learning)

print(f"Starting observation: {observation}")
# Example output: [ 0.01234567 -0.00987654  0.02345678  0.01456789]
# [cart_position, cart_velocity, pole_angle, pole_angular_velocity]

episode_over = False
total_reward = 0

while not episode_over:
    # Choose an action: 0 = push cart left, 1 = push cart right
    action = env.action_space.sample()  # Random action for now - real agents will be smarter!

    # Take the action and see what happens
    observation, reward, terminated, truncated, info = env.step(action)

    # reward: +1 for each step the pole stays upright
    # terminated: True if pole falls too far (agent failed)
    # truncated: True if we hit the time limit (500 steps)

    total_reward += reward
    episode_over = terminated or truncated

print(f"Episode finished! Total reward: {total_reward}")
env.close()

**Explaining the Code Step by Step**

First, an environment is created using `make()` with an optional `render_mode parameter that specifies how the environment should be visualized. The render mode determines whether you, among other options, see a visual window (`"human"`), or run without visuals (`None` - fastest for training).

After initializing the environment, we reset the environment with `env.reset()` to get the first observation along with additional information. This is like starting a new game or episode. 

As we want to continue the agent-environment loop until the environment ends (which happens in an unknown number of timesteps), we define `episode_over` as a variable to control our while loop.

Next, the agent performs an action in the environment. `env.step()` executes the selected action (in our example, random with `env.action_space.sample()`) to update the environment. This action can be imagined as moving a robot, pressing a button on a game controller, or making a trading decision. 

As a result, the agent receives a new observation from the updated environment along with a `reward` for taking the action. This reward could be positive for good actions (like successfully balancing the pole) or negative for bad actions (like letting the pole fall). One such action-observation exchange is called a timestep.

However, after some timesteps, the environment may end - this is called the terminal state. For instance, the robot may have crashed, or succeeded in completing a task, or we may want to stop after a fixed number of timesteps. In `Gymnasium`, if the environment has terminated due to the task being completed or failed, this is returned by `step()` as `terminated=True`. If we want the environment to end after a fixed number of timesteps, the environment issues `truncated=True`. If either `terminated` or `truncated` are `True`, the episode ends. 

As the agent is untrained yet, so it cannot make more than a couple of steps. 
We run a few episodes (= we try it again and again) and report the rewards achieved:

In [ ]:
# Create our training environment - a cart with a pole that needs balancing
env = gym.make("CartPole-v1", render_mode="human")

# Reset environment to start a new episode
observation, info = env.reset()

scores = []
episodes = 30
for episode in range(1, episodes+1):
#    observation, info = env.reset()

    state = env.reset()
    done = False
    score = 0 
    while not done:
        env.render()
        action = random.choice([0,1])
        n_state, reward, done, info, x = env.step(action)
        score+=reward
    #print('Episode:{} Score:{}'.format(episode, score))
    #print('episode ', episode)
    #print('score ', score)
    scores.append(score)

env.close()
plt.plot(range(episode), scores)
plt.title('Cumulative reward for each episode')
plt.ylabel('Cumulative reward')
plt.xlabel('episode')
plt.rcParams["figure.figsize"] = (27,8)
plt.show()

### 2.2 Training an Agent

After we have seen how the environment works in `gymnasium`, we are now ready to replace random actions with intelligence by training an agent.

In reinforcement learning, an agent is trained by remembering good decisions.

Unlike supervised learning where we show examples of correct answers, reinforcement learning agents learn by trying different actions and observing the results. It is similar to learning how to ride a bike - you try different movements, fall down a few times, and gradually learn how it works.
The goal is to develop a policy - a strategy that tells the agent what action to take in each situation to maximize long-term rewards.
This process is called **Q-Learning**.


Q-learning builds a Q-table which can be seen as a giant “cheat sheet” that tells the agent how good each action is in each situation:

* Rows: different situations (states) the agent can encounter
* Columns: different actions the agent can take
* Values: how good that action is in that situation (expected future reward)


The Learning Process works then like the following:

   1. Try an action and see what happens (reward + new state)

   2. Update your cheat sheet: “That action was better/worse than I thought”

   3. Gradually improve by trying actions and updating estimates

   4. Balance exploration vs exploitation: Try new things vs use what you know works

**Why it works:** Over time, good actions get higher Q-values, bad actions get lower Q-values. The agent learns to pick actions with the highest expected rewards.

We continue now with the actual implementation of an Reinforcement Learning agent.

We start by loading the environment.

In [27]:
env = gym.make("CartPole-v1")

For our Q-Learning model to work, we discretize the cart-pole states into finite categories. 
This simplification enables a manageable Q-table size, allowing the model to learn and optimize actions effectively within a discrete decision space.

In [13]:
# Define the number of buckets for each state dimension
n_buckets = (10, 10, 10, 10)

# Define the bounds for each state dimension
lower_bounds = [-4.8, -3.0, -0.418, -math.radians(50)]
upper_bounds = [4.8, 3.0, 0.418, math.radians(50)]

# Calculate the width of each bucket
bucket_width = [(upper_bounds[i] - lower_bounds[i]) / (n_buckets[i] - 1) for i in range(4)]

def discretizer(cart_position, cart_velocity, pole_angle, pole_angular_velocity):
    """
    Converts continuous cart-pole state variables into discrete indexes based on predefined buckets.

    Parameters:
    - cart_position (float): The cart's position.
    - cart_velocity (float): The cart's velocity.
    - pole_angle (float): The pole's angle with respect to vertical.
    - pole_angular_velocity (float): The pole's angular velocity.

    Returns:
    Tuple[int, ...]: A tuple of discrete indexes for each input state variable.
    """
    cart_pos_index = int(min(max((cart_position - lower_bounds[0]) / bucket_width[0], 0), n_buckets[0] - 1))
    cart_vel_index = int(min(max((cart_velocity - lower_bounds[1]) / bucket_width[1], 0), n_buckets[1] - 1))
    pole_angle_index = int(min(max((pole_angle - lower_bounds[2]) / bucket_width[2], 0), n_buckets[2] - 1))
    pole_vel_index = int(min(max((pole_angular_velocity - lower_bounds[3]) / bucket_width[3], 0), n_buckets[3] - 1))

    return (cart_pos_index, cart_vel_index, pole_angle_index, pole_vel_index)

Initializing the Q-table with random values instead of zeros encourages initial exploration of the action space, preventing early bias towards any specific action.

In [14]:
Q_table = np.random.uniform(low=0, high=1, size=n_buckets + (env.action_space.n,))

We define the policy function to balance exploration and exploitation.

In [15]:
def policy(state, epsilon=0.0):
    """
    Determines the action to take in a given state using an epsilon-greedy strategy.

    Parameters:
    - state: The current state of the environment.
    - epsilon (float, optional): The probability of choosing a random action for exploration.

    Returns:
    The action chosen based on the epsilon-greedy strategy.
    """
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample()
    else:
        return np.argmax(Q_table[state])    

We define the function to update Q-values, enhancing reward optimization by considering future returns.

In [16]:
def new_Q_value(reward, new_state, gamma):
    """
    Updates the Q-value for a state-action pair by incorporating future rewards.

    Parameters:
    - reward: The immediate reward received from taking an action.
    - new_state: The state transitioned into after taking the action.
    - gamma: The discount factor for future rewards.

    Returns:
    The updated Q-value incorporating future potential rewards.
    """
    future_optimal_value = np.max(Q_table[new_state])  # max_a Q(S', a)
    return reward + gamma * future_optimal_value

During training, extra rewards and punishments are given to train the agent.

We defined our reward and punishment system in 3 stages:

The first stage is based around logical deductions on what actions would be appropriate at a given state. This would allow to quickly learn what actions are good and bad.

The second stage is based around incentivising the AI to remain in an optimal zone for position, velocity, pole angle and angular velocity. This portion builds on the first portion and fine-tunes the robot to be able to balance well, rewarding stability in all 4 factors.

The last stage heavily punishes the robot if it was far from the center and still moving away from the center. This is due to the first two stages paying more focus to the balancing of the pole, which resulted in cases where the AI avoids changing directions due to the short-term instability and insteads crashes into the environment's boundaries.

In [17]:
def calculate_reward(action, current_obs, next_obs):
  reward = 0
  if action == 0 and current_obs[0] > 0.5: #if cart is close to right and action is to move left, give reward
    reward = reward + 3
  if action == 1 and current_obs[0] < -0.5:
    reward = reward + 3

  if action == 0 and current_obs[1] > 0.25: #if velocity moves right and action is to move left, give reward
    reward = reward + 5
  if action == 1 and current_obs[1] < -0.25:
    reward = reward + 5

  if action == 0 and current_obs[2] > 0: #if pole angle is leaning right and action is to move left, give punishment
    reward = reward - 10
  if action == 1 and current_obs[2] < 0:
    reward = reward - 10

  if action == 0 and current_obs[3] > 0.5: #if pole speed is moving right and action is to move left, give punishment
    reward = reward - 7
  if action == 1 and current_obs[3] < -0.5:
    reward = reward - 7

  if -0.25 < next_obs[0] < 0.25: #if cart is close to center, give reward
    reward = reward + 1
  elif -0.75 < next_obs[0] < 0.75:
    reward = reward + 0
  else:
    reward = reward - 1

  if -0.1 < next_obs[1] < 0.1: # if velocity is small, give reward
    reward = reward + 1
  elif -0.25 < next_obs[1] < 0.25:
    reward = reward + 0
  else:
    reward = reward - 1

  if -0.05 < next_obs[2] < 0.05: # if pole angle is small (in other words, pole is not leaning much), give reward
    reward = reward + 5
  elif -0.1 < next_obs[2] < 0.1:
    reward = reward + 0
  else:
    reward = reward - 5

  if -0.5 < next_obs[3] < 0.5: #if pole's rotation is small, give reward
    reward = reward + 5
  elif -1 < next_obs[3] < 1:
    reward = reward + 0
  else:
    reward = reward - 5

  #If too far too the left and velocity towards left, give punishment
  if next_obs[0] < -1 and next_obs[1] < 0:
    reward = reward - 5

  #If too far too the right and velocity towards right, give punishment
  if next_obs[0] > 1 and next_obs[1] > 0:
    reward = reward - 5
  return reward    

We have now defined all helper functions.

The following code block trains our model.
Training the model involves running it through a series of episodes to learn the optimal actions at different states under a given policy. 

The process starts with high exploration (high epsilon) to explore a wide range of actions, gradually shifting towards exploitation (lower epsilon) as the model learns from the environment. Parameters like alpha (learning rate), gamma (discount factor for future rewards), and the dynamics of epsilon (for balancing exploration and exploitation) are set to guide this learning process. Throughout each episode, the agent's observations and actions are used to update the Q-table values, continuously refining the policy towards optimal action selection based on the observed states.

In [18]:
alpha = 0.1  # Fixed step size
gamma = 1  # Discount factor
epsilon_start = 1.0  # Starting value of epsilon
epsilon_decay = 0.999  # Decay rate of epsilon after each episode
epsilon_min = 0.01  # Minimum value of epsilon

n_episodes = 1000 #10000
for e in range(n_episodes):
    epsilon = max(epsilon_min, epsilon_start * (epsilon_decay ** e))  # Decrease epsilon
    current_obs, done = env.reset(), False

    current_obs = current_obs[0]
    time = 0

    while not done:
        #current_state = discretizer(*current_obs)
        current_state = discretizer(current_obs[0],current_obs[1],current_obs[2],current_obs[3])
        action = policy(current_state, epsilon)  # Use the epsilon-greedy policy
        next_obs, reward, done, _,x = env.step(action)
        new_state = discretizer(*next_obs)
        time = time + 1
        reward = calculate_reward(action, current_obs, next_obs)

        if time > 500:
          done = True

        # Update Q-Table using the formula from the uploaded algorithm
        old_value = Q_table[current_state][action]
        learnt_value = new_Q_value(reward, new_state, gamma)
        Q_table[current_state][action] = (1 - alpha) * old_value + alpha * learnt_value

        current_obs = next_obs  # Update the observation

env.close()

### The effectiveness of the Reinforcement Learning agent

We have now trained the agent. How well does it work?

For this task, use the agent we just have trained and let it play the game for 100 episodes. While doing so, we record the cumulative reward for each round, and plot the reward for each round.

In [ ]:
episode_results = np.array([])

for _ in range(100):

  observation = env.reset()
  cumulative_reward = 0
  done = False
  observation = observation[0]
  while not done:
      
      #action = policy(discretizer(*observation))
      action = policy(discretizer(observation[0],observation[1],observation[2],observation[3]))
      observation, reward, done, info,x = env.step(action)
      cumulative_reward += reward

  episode_results = np.append(episode_results, cumulative_reward)

plt.plot(episode_results)
plt.title('Cumulative reward for each episode')
plt.ylabel('Cumulative reward')
plt.xlabel('episode')
plt.show()

We see that the learning process is highly effective!



## 3. Custom `gymnasium` Environments <a class="anchor" id="third-bullet"></a>

The included Environments are ideal for understanding Reinforcement Learning.

In many applications, however, we would need a custom Environment, in order to simulate e.g. the behavior of a robot, or the behavior of another system that makes decisions.

Here we will see how we can build custom Reinforcement Learning Environments with OpenAI `gymnasium`.

### Before You Code: Environment Design

Creating an RL environment is like designing a video game or simulation. Before writing any code, you need to think through the learning problem you want to solve. This design phase is crucial - a poorly designed environment will make learning difficult or impossible, no matter how good your algorithm is.
Key Design Questions

Ask yourself these fundamental questions:

**What skill should the agent learn?**

Navigate through a maze?

Balance and control a system?

Optimize resource allocation?

Play a strategic game?

**What information does the agent need?**

Position and velocity?

Current state of the system?

Historical data?

Partial or full observability?

**What actions can the agent take?**

Discrete choices (move up/down/left/right)?

Continuous control (steering angle, throttle)?

Multiple simultaneous actions?


**How do we measure success?**

Reaching a specific goal?

Minimizing time or energy?

Maximizing a score?

Avoiding failures?

**When should episodes end?**

Task completion (success/failure)?

Time limits?

Safety constraints?







### Our custom environment

Our custom environment is about how to keep a temperature within a narrow range.

We begin by creating a `CustomEnv` class. By passing `Env` to the `CustomEnv` class, we **inherit** the methods and properties from the OpenAI `gymnasion` environment class.

Within the `CustomEnv`class, we implement the `__init__` function to initialize the actions, observations, and episode length.
The actions are: down (`0`), keep(`1`), up (`2`).

The `observation_space` will hold an array of our current temperature. Next, we set our start temperature to 38 degrees plus a random integer. Finally, we’ve set the shower length to 60 seconds.

Other than `Discrete` spaces, `Box` spaces are much more flexible and allow us to pass through multiple values between 0 and 100. In addition, they can be used to hold other data such as images, audio, and data frames.

The `step` function defines what we do after we take action. We’ve set our action value to `-1`. Ideally, this means that:

 *   If we apply action `0` together with `-1`, we get a `-1` value. This action will lower the temperature by 1.
 *   If we apply action `1` together with `-1`, we get a `0` value. This action will maintain the current temperature.
 *   If we apply action `2` together with `-1`, we get a `1` value. This action will increase the temperature by 1.

Each step, We are also reducing the remaining shower length by 1.

When calculating the **reward**:

* If the temperature is in its optimal range of 37, and 39, we give a reward of 1.
* If the temperature is not in the optimal range, we give a reward of `-1`. 

Our model will always try to converge with this function so that the temperature is within the optimal range.


We use the `reset` function to reset our environment or update each episode. It resets the shower temperature and time.


In [3]:
import gymnasium as gym
import numpy as np
import random


class CustomEnv(gym.Env):
    def __init__(self):
        self.action_space = gym.spaces.Discrete(3)
        self.observation_space = gym.spaces.Box(low=np.array([0]), high=np.array([100]))
        self.state = 38 + random.randint(-3,3)
        self.shower_length = 60
        
    def step(self, action):
        self.state += action -1 
        self.shower_length -= 1 
        
        if self.state >=37 and self.state <=39: 
            reward =1 
        else: 
            reward = -1 
        
        if self.shower_length <= 0: 
            done = True
        else:
            done = False
        
        info = {}
        
        # Return step information
        return self.state, reward, done, info
    
    def reset(self):
        self.state = 38 + random.randint(-3,3)
        self.shower_length = 60 
        return self.state

In [4]:
env = CustomEnv()

In [5]:
env.observation_space.sample()

array([34.567307], dtype=float32)

In [19]:
env.action_space.sample()

1

Let’s play around with our environment without doing any training. We're just sampling:

In [6]:
episodes = 20 #20 shower episodes
for episode in range(1, episodes+1):
    state = env.reset()
    done = False
    score = 0 
    
    while not done:
        action = env.action_space.sample()
        n_state, reward, done, info = env.step(action)
        score+=reward
    print('Episode:{} Score:{}'.format(episode, score))

Episode:1 Score:-56
Episode:2 Score:-32
Episode:3 Score:6
Episode:4 Score:-12
Episode:5 Score:-38
Episode:6 Score:0
Episode:7 Score:-30
Episode:8 Score:-28
Episode:9 Score:-52
Episode:10 Score:-38
Episode:11 Score:-32
Episode:12 Score:-22
Episode:13 Score:-58
Episode:14 Score:-58
Episode:15 Score:-60
Episode:16 Score:-60
Episode:17 Score:-52
Episode:18 Score:-60
Episode:19 Score:-10
Episode:20 Score:-54


After running through 20 different showers, we get different reward values. 
Remember, if the shower is not within the optimal range of between 37 and 39 degrees, we get a reward of `-1`.

Most of the rewards indicate that we were way outside our optimal temperature range.
The best reward typically between 25 and 30, which indicates that some of the steps that we took may have been within that optimal range.



After this, we would proceed like in the first example: training the model, and after training, we can test it.


This is an idealized example and might not represent a real-case scenario, i.e., when something else is influencing with the temperature. It is thus always important to build a model as close as possible to the scenario in case.

More on custom environments:

https://gymnasium.farama.org/introduction/create_custom_env/

## 4. Summary <a class="anchor" id="fourth-bullet"></a>

We have seen how Reinforced Learning can provide a way of training a model when supervised learning by examples is not possible.

Custom environments within the framework of `gymnasium` allow to build and train these models.



